# 02. vae.py: компактный язык кадров

VAE превращает пиксели в непрерывные латенты. Разберём causal convolution, сжатие времени, posterior, scaling и tiling. KVAE вынесен в отдельную лабораторную: выбранный distill использует HunyuanVideo VAE.


### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


## Часть A. Теория VAE: от сжатия к вероятностной модели

После этой части вы сможете вывести ELBO, отличить prior от posterior, написать differentiable sampling и объяснить, почему готовый VAE и Flow Matching учатся разным loss.

**Обозначения:** $x$ — наблюдаемые пиксели, $z$ — скрытый непрерывный код; $\phi$ — веса encoder, $\theta$ — веса decoder. $q_\phi(z\mid x)$ — приближённый posterior, $p(z)$ — prior, $p_\theta(x\mid z)$ — модель наблюдения. Вероятностное условие $\mid$ читается «при известном».

Обычный autoencoder кодирует картинку одной точкой. VAE кодирует её распределением: центр говорит «какой код подходит», разброс — «насколько можно его менять». Decoder должен восстанавливать изображение из соседних точек, а не только из одного запомненного кода. Это полезная интуиция; математическое основание — вероятностная цель ниже.

### A1. Откуда берётся задача обучения

Порождающая модель: сначала $z\sim p(z)=\mathcal N(0,I)$, затем $x\sim p_\theta(x\mid z)$. Хотим увеличить likelihood наблюдаемых данных:

$$p_\theta(x)=\int p_\theta(x\mid z)p(z)\,dz.$$

Интеграл суммирует объяснения картинки всеми возможными кодами. При нейросетевом decoder его обычно нельзя вычислить аналитически. Encoder не заменяет этот интеграл: он даёт распределение, из которого удобно брать подходящие $z$ для обучения.

**Bayes:** $p_\theta(z\mid x)=p_\theta(x\mid z)p(z)/p_\theta(x)$. Это истинный posterior; он тоже зависит от трудного знаменателя. Поэтому учим доступное $q_\phi(z\mid x)$, обычно Gaussian с diagonal covariance.

### A2. Вывод ELBO по шагам

Умножим и разделим под интегралом на $q_\phi$ (при подходящей поддержке распределений):

$$\log p_\theta(x)=\log\mathbb E_{q_\phi(z\mid x)}\left[\frac{p_\theta(x\mid z)p(z)}{q_\phi(z\mid x)}\right].$$

Логарифм — вогнутая функция. Jensen позволяет перенести его внутрь ожидания и получить **нижнюю границу**:

$$\log p_\theta(x)\geq\mathbb E_q[\log p_\theta(x\mid z)]-D_{KL}(q_\phi(z\mid x)\parallel p(z))=\mathrm{ELBO}.$$

Точнее, $\log p_\theta(x)=\mathrm{ELBO}+D_{KL}(q_\phi(z\mid x)\parallel p_\theta(z\mid x))$. Второй KL — зазор до истинного posterior; первый KL внутри ELBO — регуляризация к prior. **Это два разных KL.**

Минимизируем negative ELBO: reconstruction negative log-likelihood + KL. Реконструкция требует информативный код; KL ограничивает, как сильно posterior отклоняется от prior. [Первичная статья VAE](https://arxiv.org/abs/1312.6114).


### A3. Небольшой exact пример: проверяем нижнюю границу

Возьмём линейную Gaussian модель: $z\sim N(0,1)$, $x\mid z\sim N(z,\sigma_x^2)$. Тогда $x\sim N(0,1+\sigma_x^2)$, а истинный posterior имеет mean $x/(1+\sigma_x^2)$ и variance $\sigma_x^2/(1+\sigma_x^2)$.

Здесь интеграл известен: можно проверить ELBO численно без Monte Carlo. Для плохого $q$ ELBO ниже log likelihood; для точного posterior граница совпадает. Decoder в этом опыте не нейросеть: мы изолировали одну математическую идею.


In [ ]:
x_obs=1.3; obs_var=0.25
true_mean=x_obs/(1+obs_var); true_var=obs_var/(1+obs_var)
log_evidence=-0.5*(np.log(2*np.pi*(1+obs_var))+x_obs**2/(1+obs_var))
def exact_elbo(mean,var):
    expected_nll=0.5*(np.log(2*np.pi*obs_var)+((x_obs-mean)**2+var)/obs_var)
    kl_prior=0.5*(mean**2+var-1-np.log(var))
    return -expected_nll-kl_prior
for name,mean,var in [('bad q',0.,1.),('exact posterior',true_mean,true_var)]:
    bound=exact_elbo(mean,var)
    kl_gap=0.5*(var/true_var+(mean-true_mean)**2/true_var-1+np.log(true_var/var))
    print(name,'ELBO:',round(bound,5),'log p(x):',round(log_evidence,5),'gap:',round(kl_gap,5))
    assert bound<=log_evidence+1e-7
    assert np.isclose(log_evidence-bound,kl_gap)


### A4. Diagonal Gaussian: почему encoder выдаёт 2C каналов

$q_\phi(z\mid x)=N(\mu_\phi(x),\operatorname{diag}(\sigma^2_\phi(x)))$. Encoder выдаёт mean и **log variance**, а не mean и standard deviation: $\ell=\log\sigma^2$, $\sigma=\exp(\ell/2)$. Так variance остаётся положительной.

$$D_{KL}(q\parallel N(0,I))=\frac12\sum_j(\mu_j^2+\exp\ell_j-1-\ell_j).$$

Это сумма по latent coordinates одного примера, затем обычно среднее по batch. Смена `sum` на `mean` меняет относительную силу KL. У Hunyuan 16 latent channels, но posterior parameters имеют **32 канала**. Это continuous KL-VAE; название `quant_conv` не означает discrete VQ codebook.


In [ ]:
mu=torch.tensor([[0.2,-0.3]]); logvar=torch.tensor([[-1.,0.5]])
manual=0.5*(mu.square()+logvar.exp()-1-logvar).sum(-1)
q=torch.distributions.Normal(mu,(0.5*logvar).exp())
p=torch.distributions.Normal(torch.zeros_like(mu),torch.ones_like(mu))
api=torch.distributions.kl_divergence(q,p).sum(-1)
assert torch.allclose(manual,api)
print('KL per example:',manual,'exact q=prior KL:',0.5*(torch.zeros(2)+torch.ones(2)-1).sum())


### A5. Reparameterization: как градиент проходит через случайность

Пишем $\epsilon\sim N(0,I)$ и $z=\mu+\exp(\ell/2)\epsilon$. Шум независим от весов; $z$ — differentiable функция mean/logvar. `Normal.sample()` обычно не сохраняет нужный путь gradient; `rsample()` реализует reparameterization.

Для одного фиксированного noise sample $\partial z/\partial\mu=1$, $\partial z/\partial\ell=\frac12\sigma\epsilon$. Проверим autograd. Зажим logvar предотвращает переполнение; слишком сильное clipping меняет модель.


In [ ]:
mu=torch.tensor([0.2,-0.3],requires_grad=True)
lv=torch.tensor([-1.,0.5],requires_grad=True); eps=torch.tensor([0.4,-0.8])
std=(lv/2).exp(); z=mu+std*eps
z.sum().backward()
assert torch.allclose(mu.grad,torch.ones_like(mu))
assert torch.allclose(lv.grad,0.5*std.detach()*eps)
print('mean gradient:',mu.grad,'logvar gradient:',lv.grad)
dist=torch.distributions.Normal(mu,std)
print('sample differentiable:',dist.sample().requires_grad,'rsample:',dist.rsample().requires_grad)


### A6. Как reconstruction loss связан с вероятностью

При Gaussian decoder с фиксированной observation variance NLL пропорционален squared error. При Bernoulli decoder для binary pixels получаем binary cross-entropy. Выбор **не взаимозаменяем**: Bernoulli хорош для учебных бинарных фигур, но не утверждается как loss Hunyuan.

$L_\beta=\mathbb E_q[-\log p_\theta(x\mid z)]+\beta KL$. При $\beta=1$ и согласованном likelihood/reduction это negative ELBO; произвольный $\beta$ — другая регуляризованная цель. Предыдущий простой MiniVAE с mean-MSE + 0.001·mean-KL будет лишь наглядной аппроксимацией. Ниже добавим loss с явными единицами **на один пример**.


In [ ]:
x=torch.tensor([[0.,1.,1.,0.]])
logits=torch.tensor([[-1.,1.,0.3,-0.5]])
bernoulli_nll=torch.nn.functional.binary_cross_entropy_with_logits(logits,x,reduction='none').sum(-1)
prob=torch.sigmoid(logits)
explicit=-(x*prob.log()+(1-x)*(1-prob).log()).sum(-1)
assert torch.allclose(bernoulli_nll,explicit)
print('Bernoulli NLL per image:',bernoulli_nll)


## Часть B. Базовые опыты и связь с исходниками

### 1. Почему генерировать в латентах дешевле

Вместо RGB [B,3,T,H,W] DiT работает с [T,H/8,W/8,16]. Пространственные размеры меньше, но каналов больше. Формат DiT channels-last, формат VAE channels-first. permute меняет расположение осей, reshape не является заменой permute.


In [ ]:
pixels=torch.zeros(1,3,9,64,64)
z=torch.zeros(1,16,3,8,8)
print('RGB:',pixels.shape,'latent:',z.shape)
print('DiT layout:',z[0].permute(1,2,3,0).shape)


### 2. Причинность: будущее не должно менять прошлое

Обычная симметричная convolution смотрит в обе стороны времени. Causal convolution добавляет padding только слева по времени. Ниже меняем последний кадр и убеждаемся, что первые выходы не изменились.


In [ ]:
import torch.nn.functional as F
conv=torch.nn.Conv3d(1,1,(3,1,1),bias=False)
x=torch.arange(5.).reshape(1,1,5,1,1)
def causal(x): return conv(F.pad(x,(0,0,0,0,2,0),mode='replicate'))
y=causal(x); changed=x.clone(); changed[:,:,4]=999
print(y.flatten()); print(causal(changed).flatten())
assert torch.allclose(y[:,:,:4],causal(changed)[:,:,:4])


### 3. Вариационность простыми словами

Encoder предсказывает центр μ и разброс σ для каждого латента. mode выбирает центр; sample добавляет шум μ+σ·ε. KL штраф удерживает распределение близко к нормальному. Наша формула диагонального posterior — механизм VAE; это не переобучение Hunyuan.


In [ ]:
mu=torch.tensor([0.2,-0.3]); logvar=torch.tensor([-1.,0.5])
std=(0.5*logvar).exp()
z_sample=mu+std*torch.randn_like(std)
kl=0.5*(mu.square()+logvar.exp()-1-logvar).sum()
print('mode:',mu,'sample:',z_sample,'KL:',kl.item())


### 4. Мини-VAE: encoder, posterior, decoder

Создадим крошечную сеть для 8×8 картинок. Encoder сжимает 64 числа в 2 координаты, decoder восстанавливает 64. Здесь учатся именно VAE-веса. При обучении Kandinsky эти веса заменяются готовым внешним VAE, а обучается DiT.


In [ ]:
class MiniVAE(torch.nn.Module):
    def __init__(self):
        super().__init__(); self.encoder=torch.nn.Sequential(torch.nn.Linear(64,32),torch.nn.SiLU())
        self.mean=torch.nn.Linear(32,2); self.logvar=torch.nn.Linear(32,2)
        self.decoder=torch.nn.Sequential(torch.nn.Linear(2,32),torch.nn.SiLU(),torch.nn.Linear(32,64),torch.nn.Sigmoid())
    def forward(self,x):
        h=self.encoder(x); mu=self.mean(h); lv=self.logvar(h).clamp(-8,4)
        z=mu+torch.exp(lv/2)*torch.randn_like(mu)
        return self.decoder(z),mu,lv
data=torch.zeros(32,8,8)
for i in range(32): data[i,2:6,i%5:i%5+3]=1
flat=data.flatten(1); mini=MiniVAE(); opt=torch.optim.Adam(mini.parameters(),lr=0.01)
losses=[]
for step in range(180):
    rec,mu,lv=mini(flat)
    recon_loss=F.mse_loss(rec,flat); kl_loss=0.5*(mu.square()+lv.exp()-1-lv).mean()
    loss=recon_loss+0.001*kl_loss
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
plt.plot(losses); plt.xlabel('training step'); plt.ylabel('reconstruction + KL'); plt.show()


### 5. Как читать результат

Нельзя оценивать генеративную модель только по train loss. Сначала проверим реконструкцию mode: два латентных числа явно не сохранят все детали. Затем сравните sample, чтобы увидеть влияние posterior noise.


In [ ]:
with torch.no_grad(): restored=mini.decoder(mini.mean(mini.encoder(flat))).reshape(-1,8,8)
fig,axes=plt.subplots(1,2); axes[0].imshow(data[2]); axes[0].set_title('input')
axes[1].imshow(restored[2]); axes[1].set_title('posterior mode'); plt.show()


### 6. Архитектура настоящего vae.py

CausalConv3d → ResnetBlockCausal3D → DownBlock3D → Encoder3D → posterior → Decoder3D → UpBlock3D. MidBlock3D содержит attention на сжатом уровне. UpsampleCausal3D отдельно обрабатывает первый кадр. AutoencoderKLHunyuanVideo объединяет сеть и tiling. build_vae выбирает Hunyuan для video или AutoencoderKL FLUX для image.


In [ ]:
source_excerpt('kandinsky/models/vae.py','def build_vae',17)
source_excerpt('kandinsky/models/vae.py','class HunyuanVideoCausalConv3d',40)


### 7. Scaling — часть договора DiT/VAE

При encoding латент умножают на scaling_factor, перед decoding делят на него. Для Hunyuan в выбранном T2V нет FLUX shift. Неверный scale меняет распределение, которое видит decoder. Проверим обратимость на игрушечном примере.


In [ ]:
raw=torch.randn(1,16,3,8,8); scaling_factor=0.476986
scaled=raw*scaling_factor
assert torch.allclose(raw,scaled/scaling_factor,atol=1e-6)
print('σ raw:',raw.std().item(),'σ scaled:',scaled.std().item())


### 8. Настоящая реконструкция Hunyuan: только VAE

Включение скачает только vae/*, не DiT и не Qwen. CPU FP32 — проверяемая базовая ветка для Conv3d на Mac. Видео 9×64×64 создано локально. Сначала измеряем качество mode на том же изображении, затем исследуем tiling. PSNR измеряет ошибку пикселей, а не эстетическое качество.


In [ ]:
RUN_REAL_VAE=False
if RUN_REAL_VAE:
    from labkit.components import download_component,load_video_vae
    download_component('vae'); vae=load_video_vae(device='cpu')
    video=torch.full((1,3,9,64,64),-1.)
    for t in range(9): video[:,:,t,20:40,5+t*3:21+t*3]=1
    with torch.inference_mode():
        posterior=vae.encode(video).latent_dist
        latent=posterior.mode(); reconstruction=vae.decode(latent).sample.clamp(-1,1)
    print('latent:',latent.shape,'reconstructed:',reconstruction.shape,'scale:',vae.config.scaling_factor)
    mse=F.mse_loss((reconstruction+1)/2,(video+1)/2).item()
    print('PSNR:',-10*np.log10(max(mse,1e-12)))
    plt.imshow((reconstruction[0,:,4].permute(1,2,0)+1)/2); plt.show()
    del vae,posterior; release()
else: print('Настоящий VAE пропущен.')


### 9. Tiling и ограничения

Tiling декодирует перекрывающиеся плитки и смешивает края. Это уменьшает пик памяти, но может изменить результат на границах. В upstream get_enc_optimal_tiling обращается к torch.cuda.mem_get_info: для MPS он непригоден. Наш Mac путь использует diffusers decoder с его tiling и CPU FP32. Это адаптация, не доказанная побитовая эквивалентность.


In [ ]:
source_excerpt('kandinsky/models/vae.py','def blend_v',15)
source_excerpt('kandinsky/models/vae.py','def get_enc_optimal_tiling',17)


#


In [ ]:
#


## Часть C. Более строгий учебный VAE: сравниваем β, prior и posterior

Для 8×8 фигур обучим encoder → mean/logvar → reparameterized latent → decoder logits. Суммируем reconstruction по **64 пикселям**, KL по **2 координатам**, затем усредняем batch. Две модели начинают с одинаковых весов и обучаются на одном наборе данных; сравнение β остаётся малым педагогическим экспериментом, а не benchmark.

Отдельно сохраняем reconstruction NLL и KL: общий loss без разложения не показывает, использует ли encoder латент. При posterior collapse $q(z\mid x)$ близок к prior и decoder может игнорировать $z$. Маленький KL сам по себе не доказывает collapse; смотрим также разброс mean между входами и зависимость decoder от z.


In [ ]:
class ELBOVAE(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=torch.nn.Sequential(torch.nn.Linear(64,48),torch.nn.SiLU(),torch.nn.Linear(48,4))
        self.decoder=torch.nn.Sequential(torch.nn.Linear(2,48),torch.nn.SiLU(),torch.nn.Linear(48,64))
    def encode(self,x):
        mu,lv=self.encoder(x).chunk(2,-1)
        return mu,lv.clamp(-10,5)
    def forward(self,x):
        mu,lv=self.encode(x)
        z=mu+torch.exp(lv/2)*torch.randn_like(mu)
        return self.decoder(z),mu,lv
binary=data.flatten(1)
models={}; histories={}
for beta in [0.01,1.0]:
    torch.manual_seed(7); network=ELBOVAE(); optimizer=torch.optim.Adam(network.parameters(),lr=0.01)
    history=[]
    for step in range(400):
        logits,mu,lv=network(binary)
        nll=F.binary_cross_entropy_with_logits(logits,binary,reduction='none').sum(-1).mean()
        kl=0.5*(mu.square()+lv.exp()-1-lv).sum(-1).mean()
        objective=nll+beta*kl
        optimizer.zero_grad(); objective.backward(); optimizer.step()
        history.append((nll.item(),kl.item()))
    models[beta]=network.eval(); histories[beta]=np.array(history)
fig,axes=plt.subplots(1,2,figsize=(10,3))
for beta,history in histories.items():
    axes[0].plot(history[:,0],label=f'beta={beta}')
    axes[1].plot(history[:,1],label=f'beta={beta}')
axes[0].set_title('Reconstruction NLL / image'); axes[1].set_title('KL / image')
for ax in axes: ax.set_xlabel('step'); ax.legend()
plt.tight_layout(); plt.show()


### C2. Реконструкция и генерация — разные операции

**Reconstruction:** $x\to q(z\mid x)\to z\to decoder$. **Prior generation:** $z\sim N(0,I)\to decoder$, без входного изображения. Posterior mode полезен для детерминированного сравнения, но это не случайная генерация. Aggregate posterior $q(z)=\mathbb E_{data}q(z\mid x)$ не обязан точно совпадать с prior даже после KL-регуляризации.

Ниже сравним input, reconstruction и prior samples; scatter posterior means покажет геометрию кодов. Если β слишком мал, reconstruction может быть хорошей, а случайные prior samples — плохими: prior посещает области, с которыми decoder мало работал.


In [ ]:
fig,axes=plt.subplots(2,6,figsize=(10,4))
for row,beta in enumerate([0.01,1.0]):
    net=models[beta]
    with torch.no_grad():
        means,lv=net.encode(binary)
        restored=net.decoder(means).sigmoid().reshape(-1,8,8)
        prior=net.decoder(torch.randn(4,2)).sigmoid().reshape(-1,8,8)
    images=[data[0],restored[0],*list(prior)]
    for col,img in enumerate(images):
        axes[row,col].imshow(img,vmin=0,vmax=1,cmap='gray'); axes[row,col].axis('off')
        axes[row,col].set_title(('input' if col==0 else 'mode' if col==1 else 'prior')+f' b={beta}')
    print('beta',beta,'variation of posterior mean:',means.std(0).tolist())
plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(4,4))
for beta,net in models.items():
    with torch.no_grad(): means,_=net.encode(binary)
    ax.scatter(means[:,0],means[:,1],label=f'beta={beta}',alpha=0.6)
ax.set_xlabel('mu_1'); ax.set_ylabel('mu_2'); ax.legend(); plt.show()


## Часть D. Прямой разбор настоящего vae.py Kandinsky

Используем **классы именно сохранённого upstream vae.py**, а не наш MiniVAE и не только библиотечный decoder. Уменьшим widths, latent channels и размеры видео; весов не скачиваем. `quant_conv` 1×1×1 смешивает 2C posterior parameters, `post_quant_conv` — C latent channels. Никакого argmin/codebook здесь нет.

Для CPU вызываем encoder/quant_conv и post_quant_conv/decoder напрямую. Публичный `decode()` upstream всегда обращается к CUDA memory planner: обход planner здесь явный и ограничен маленькой архитектурной проверкой. Структура основной сети сохраняется; полная эквивалентность tiling не утверждается.


In [ ]:
import importlib.util
path=SOURCE/'kandinsky/models/vae.py'
spec=importlib.util.spec_from_file_location('_k5_vae_theory',path)
vae_module=importlib.util.module_from_spec(spec); spec.loader.exec_module(vae_module)
small_vae=vae_module.AutoencoderKLHunyuanVideo(
    latent_channels=4,block_out_channels=(8,16,16,16),layers_per_block=1,
    norm_num_groups=4,mid_block_add_attention=False).eval()
video=torch.randn(1,3,9,32,32)
shapes={}
def shape_hook(name):
    def record(module,args,out):
        if isinstance(out,torch.Tensor): shapes[name]=tuple(out.shape)
    return record
handles=[]
for name,layer in small_vae.named_modules():
    if name in ['encoder.conv_in','encoder.down_blocks.0','encoder.down_blocks.1','encoder.down_blocks.2','encoder.down_blocks.3','encoder.conv_out','quant_conv','post_quant_conv','decoder']:
        handles.append(layer.register_forward_hook(shape_hook(name)))
with torch.no_grad():
    moments=small_vae.quant_conv(small_vae.encoder(video))
    posterior=vae_module.DiagonalGaussianDistribution(moments)
    mode=posterior.mode()
    rec=small_vae.decoder(small_vae.post_quant_conv(mode))
for h in handles: h.remove()
print('posterior parameters:',moments.shape,'latent:',mode.shape,'RGB:',rec.shape)
for name,shape in shapes.items(): print(name,'→',shape)
assert moments.shape[1]==2*mode.shape[1]
assert rec.shape==video.shape


### D2. Почему время сжимается иначе, чем пространство

Для causal codec первый кадр имеет особую роль: $T_{latent}=1+(T_{RGB}-1)/4$ для совместимых длин $T_{RGB}=4k+1$. Пространство сжимается обычными уровнями downsample. В исходном encoder при temporal compression 4 временные stride 2 стоят на промежуточных уровнях; spatial stride 2 — на первых трёх.

CausalConv3d добавляет padding только в прошлое; это свойство convolution, **не автоматическое доказательство strict causality всей сети**. Нормализации, mid attention и обработка сегментов тоже нужно проверять. GroupNorm может агрегировать по временной оси, поэтому изменение будущего потенциально влияет на нормализованные признаки. Ранее проверенный локальный conv-test проверяет только conv.


In [ ]:
for i in range(4):
    spatial=i<3
    temporal=i>=1 and i<3
    print('encoder stage',i,'stride',(2 if temporal else 1,2 if spatial else 1,2 if spatial else 1))
for frames in [1,9,121]:
    latent=(frames-1)//4+1
    print(frames,'RGB →',latent,'latent →',4*(latent-1)+1,'RGB')
source_excerpt('kandinsky/models/vae.py','if temporal_compression_ratio == 4',22)


### D3. Стыковка с DiT: VAE prior и noise sampler — не одно распределение

При T2V inference VAE encoder не нужен: DiT начинает с Gaussian noise и выдаёт **распределение data latents**, после чего VAE decoder переводит их в RGB. Encoder нужен для получения data latents при обучении и для image/video conditioning.

Важно: DiT noise $\epsilon\sim N(0,I)$ — начальная точка flow. VAE prior $p(z)=N(0,I)$ — регуляризатор codec. Финальные DiT latents не обязаны оставаться нормальными. $z_{DiT}=s z_{VAE}$; decoder получает $z_{DiT}/s$. Масштабирование не добавляет информационного содержания, но меняет численные масштабы обучения генератора.


In [ ]:
scale=small_vae.config.scaling_factor
z_dit=mode*scale
layout=z_dit[0].permute(1,2,3,0).contiguous()
back=layout.permute(3,0,1,2).unsqueeze(0)/scale
assert torch.allclose(mode,back,atol=1e-6)
print('VAE:',mode.shape,'→ DiT:',layout.shape,'scale:',scale)
source_excerpt('kandinsky/generation_utils.py','images = (images / vae.config.scaling_factor)',8)
del small_vae,posterior; release()


### D4. Что мы можем утверждать о настоящем обучении VAE

ELBO выше — общая теория. В `vae.py` есть архитектура и posterior, но нет VAE trainer и полной weighting map reconstruction/perceptual/adversarial/KL losses. Нельзя выводить исторический loss Hunyuan только из имени `AutoencoderKL`. Продвинутые visual codecs могут сочетать несколько целей; наша Bernoulli mini-задача не воспроизводит их качество.

**Задания с проверяемым результатом:**

1. Выведите KL для diagonal Gaussian на бумаге и сравните с API для 100 случайных mean/logvar.
2. Проверьте ELBO gap в exact Gaussian примере при q variance 0.01/0.2/2.
3. Повторите β experiment на новых бинарных фигурах и отдельно измерьте validation NLL/KL.
4. Разделите интерполяцию posterior means и prior sampling: почему это разные проверки?
5. Сравните whole-video и tiled reconstruction настоящих весов, измерьте ошибку на границах отдельно.
6. Проверьте воздействие изменения будущих кадров после conv, normalization и encoder целиком; не переносите результат одного слоя на всю сеть.

Источники: [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114), [HunyuanVideo](https://arxiv.org/abs/2412.03603), pinned `vae.py` в `reference`. Первые источники объясняют метод/codec; конкретные формы и API проверяем по локальному коду.


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
